# Task5 Rescued Cases Interactive Explorer (Gemma)

This notebook visualizes and explores rescued samples from:
- output/supplementary_figures/task5_top50_rescued_cases_gemma.csv
- output/supplementary_figures/task5_all_low_fail_high_success_cases_gemma.csv (for full response text)

Features:
1. Filter by prompt variant, rescue type, gold winner, human winner, model pair, and keyword.
2. Click points in scatter plot to select sample.
3. Browse one sample at a time with full details (prompt setting, verdict trajectory, response A/B, reference).

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display, HTML, Markdown

BASE_DIR = Path('..')
TOP50_PATH = BASE_DIR / 'output' / 'supplementary_figures' / 'task5_top50_rescued_cases_gemma.csv'
ALL_PATH = BASE_DIR / 'output' / 'supplementary_figures' / 'task5_all_low_fail_high_success_cases_gemma.csv'

if not TOP50_PATH.exists():
    raise FileNotFoundError(f'Missing file: {TOP50_PATH}')

df_top = pd.read_csv(TOP50_PATH)
df = df_top.copy()

if ALL_PATH.exists():
    df_all = pd.read_csv(ALL_PATH)
    keep_cols = [c for c in ['question_id', 'prompt_variant', 'answer_a_text', 'answer_b_text', 'reference_answer'] if c in df_all.columns]
    if len(keep_cols) >= 3:
        df = df.merge(df_all[keep_cols].drop_duplicates(subset=['question_id', 'prompt_variant']), on=['question_id', 'prompt_variant'], how='left')

def _pick_col(full_col, snippet_col):
    if full_col in df.columns:
        return df[full_col].fillna(df.get(snippet_col, ''))
    return df.get(snippet_col, '')

df['answer_a_show'] = _pick_col('answer_a_text', 'answer_a_text_snippet').astype(str)
df['answer_b_show'] = _pick_col('answer_b_text', 'answer_b_text_snippet').astype(str)
df['reference_show'] = _pick_col('reference_answer', 'reference_answer_snippet').astype(str)

df['rescue_type'] = np.select(
    [
        df['rescued_ensemble_only'].astype(bool),
        df['rescued_by_single'].astype(bool) & df['rescued_by_ensemble'].astype(bool),
        df['rescued_by_single'].astype(bool),
        df['rescued_by_ensemble'].astype(bool),
    ],
    [
        'Ensemble Only',
        'Both Single and Ensemble',
        'Single Only',
        'Ensemble (possibly overlap)',
    ],
    default='Other',
)

df['sample_id'] = np.arange(len(df))
df['keyword_blob'] = (
    df['question_id'].astype(str) + ' ' +
    df['prompt_variant'].astype(str) + ' ' +
    df['model_a'].astype(str) + ' ' +
    df['model_b'].astype(str) + ' ' +
    df['answer_a_show'].astype(str) + ' ' +
    df['answer_b_show'].astype(str) + ' ' +
    df['reference_show'].astype(str)
).str.lower()

print(f'Loaded samples: {len(df):,}')
print('Columns:', df.columns.tolist())

In [ ]:
overview = pd.DataFrame({
    'Metric': [
        'N samples',
        'N prompt variants',
        'N rescue types',
        'Mean entropy (high ensemble)',
        'Mean distinct verdicts (high ensemble)',
    ],
    'Value': [
        len(df),
        df['prompt_variant'].nunique(),
        df['rescue_type'].nunique(),
        df['ent_high_ens_mean'].mean(),
        df['distinct_high_ens_mean'].mean(),
    ],
})
display(overview)

fig1 = px.histogram(df, x='ent_high_ens_mean', color='prompt_variant', barmode='overlay', nbins=20,
                    title='Entropy Distribution (High Ensemble)')
fig1.update_layout(height=380)
fig1.show()

fig2 = px.bar(df['rescue_type'].value_counts().reset_index(), x='rescue_type', y='count',
              title='Rescue Type Counts')
fig2.update_layout(height=380, xaxis_title='Rescue Type', yaxis_title='Count')
fig2.show()

In [ ]:
prompt_opts = sorted(df['prompt_variant'].dropna().unique().tolist())
rescue_opts = sorted(df['rescue_type'].dropna().unique().tolist())
gold_opts = ['ALL'] + sorted(df['gold_winner'].dropna().astype(str).unique().tolist())
human_opts = ['ALL'] + sorted(df['human_winner'].dropna().astype(str).unique().tolist())
model_opts = ['ALL'] + sorted((df['model_a'].astype(str) + ' vs ' + df['model_b'].astype(str)).unique().tolist())

w_prompt = widgets.SelectMultiple(options=prompt_opts, value=tuple(prompt_opts), description='Prompt', rows=3)
w_rescue = widgets.SelectMultiple(options=rescue_opts, value=tuple(rescue_opts), description='Rescue', rows=4)
w_gold = widgets.Dropdown(options=gold_opts, value='ALL', description='Gold')
w_human = widgets.Dropdown(options=human_opts, value='ALL', description='Human')
w_model = widgets.Dropdown(options=model_opts, value='ALL', description='Model Pair')
w_keyword = widgets.Text(value='', description='Keyword', placeholder='question_id or text keyword')
w_sort = widgets.Dropdown(options=['ent_high_ens_mean', 'distinct_high_ens_mean', 'question_id'], value='ent_high_ens_mean', description='Sort by')
w_desc = widgets.Checkbox(value=True, description='Descending')
w_idx = widgets.IntSlider(value=0, min=0, max=max(len(df)-1, 0), step=1, description='Sample')

out_status = widgets.Output()
out_scatter = widgets.Output()
out_table = widgets.Output()
out_detail = widgets.Output()

def apply_filters():
    d = df.copy()
    d = d[d['prompt_variant'].isin(list(w_prompt.value))]
    d = d[d['rescue_type'].isin(list(w_rescue.value))]
    if w_gold.value != 'ALL':
        d = d[d['gold_winner'].astype(str) == w_gold.value]
    if w_human.value != 'ALL':
        d = d[d['human_winner'].astype(str) == w_human.value]
    if w_model.value != 'ALL':
        pair = d['model_a'].astype(str) + ' vs ' + d['model_b'].astype(str)
        d = d[pair == w_model.value]
    key = w_keyword.value.strip().lower()
    if key:
        d = d[d['keyword_blob'].str.contains(key, na=False)]
    d = d.sort_values(w_sort.value, ascending=not w_desc.value).reset_index(drop=True)
    return d

def render_sample(d):
    with out_status:
        out_status.clear_output(wait=True)
        print(f'Filtered samples: {len(d)}')

    if len(d) == 0:
        with out_scatter:
            out_scatter.clear_output(wait=True)
            print('No samples after filtering.')
        with out_table:
            out_table.clear_output(wait=True)
            print('No rows to display.')
        with out_detail:
            out_detail.clear_output(wait=True)
            print('No sample details.')
        w_idx.max = 0
        w_idx.value = 0
        return

    w_idx.max = len(d) - 1
    if w_idx.value > w_idx.max:
        w_idx.value = w_idx.max

    with out_scatter:
        out_scatter.clear_output(wait=True)
        fig = go.FigureWidget(
            data=[
                go.Scatter(
                    x=d['ent_high_ens_mean'],
                    y=d['distinct_high_ens_mean'],
                    mode='markers',
                    marker=dict(size=10, opacity=0.8),
                    text=('qid=' + d['question_id'].astype(str) + ', ' + d['prompt_variant'].astype(str)),
                    customdata=np.arange(len(d)),
                    hovertemplate='Entropy=%{x:.3f}<br>Distinct=%{y:.3f}<br>%{text}<extra></extra>',
                )
            ]
        )
        fig.update_layout(
            title='Click a point to select sample',
            xaxis_title='Mean Vote Entropy (High Ensemble)',
            yaxis_title='Mean Distinct Verdicts (High Ensemble)',
            height=420,
        )

        def _on_click(trace, points, selector):
            if points.point_inds:
                w_idx.value = int(points.point_inds[0])

        fig.data[0].on_click(_on_click)
        display(fig)

    with out_table:
        out_table.clear_output(wait=True)
        cols = [
            'question_id', 'prompt_variant', 'rescue_type', 'gold_winner', 'human_winner',
            'v_low_sp', 'v_high_sp_1.5', 'v_high_sp_2.0', 'v_high_sp_3.0',
            'v_high_ens_1.5', 'v_high_ens_2.0', 'v_high_ens_3.0',
            'ent_high_ens_mean', 'distinct_high_ens_mean',
        ]
        display(d[cols].head(200))

    row = d.iloc[w_idx.value]
    verdict_tbl = pd.DataFrame({
        'Setting': ['Low SP', 'High SP 1.5', 'High SP 2.0', 'High SP 3.0', 'High ENS 1.5', 'High ENS 2.0', 'High ENS 3.0'],
        'Verdict': [
            row.get('v_low_sp', np.nan),
            row.get('v_high_sp_1.5', np.nan),
            row.get('v_high_sp_2.0', np.nan),
            row.get('v_high_sp_3.0', np.nan),
            row.get('v_high_ens_1.5', np.nan),
            row.get('v_high_ens_2.0', np.nan),
            row.get('v_high_ens_3.0', np.nan),
        ],
    })

    with out_detail:
        out_detail.clear_output(wait=True)
        header = f"""
        <h3>Sample Detail: question_id={row['question_id']} | prompt={row['prompt_variant']}</h3>
        <b>Rescue Type:</b> {row['rescue_type']}<br>
        <b>Gold Winner:</b> {row['gold_winner']} | <b>Human Winner:</b> {row['human_winner']}<br>
        <b>Model A:</b> {row['model_a']}<br>
        <b>Model B:</b> {row['model_b']}<br>
        <b>Entropy Mean:</b> {row['ent_high_ens_mean']:.4f} | <b>Distinct Verdict Mean:</b> {row['distinct_high_ens_mean']:.4f}
        """
        display(HTML(header))
        display(verdict_tbl)

        display(Markdown('### Response A'))
        display(HTML(f"<div style='white-space: pre-wrap; border: 1px solid #ddd; padding: 8px;'>{row['answer_a_show']}</div>"))

        display(Markdown('### Response B'))
        display(HTML(f"<div style='white-space: pre-wrap; border: 1px solid #ddd; padding: 8px;'>{row['answer_b_show']}</div>"))

        display(Markdown('### Reference / Gold Explanation'))
        display(HTML(f"<div style='white-space: pre-wrap; border: 1px solid #ddd; padding: 8px;'>{row['reference_show']}</div>"))

def refresh(_=None):
    d = apply_filters()
    render_sample(d)

for w in [w_prompt, w_rescue, w_gold, w_human, w_model, w_keyword, w_sort, w_desc, w_idx]:
    w.observe(refresh, names='value')

control_box = widgets.VBox([
    widgets.HBox([w_prompt, w_rescue]),
    widgets.HBox([w_gold, w_human, w_model]),
    widgets.HBox([w_keyword, w_sort, w_desc]),
    w_idx,
    out_status,
])

display(control_box)
display(out_scatter)
display(out_table)
display(out_detail)

refresh()

## Usage Tips

1. Start with prompt filter to isolate baseline or cot.
2. Use rescue filter to focus on ensemble-only or single-only recovery.
3. Enter keyword (for example, theorem, physics, legal) to inspect domain slices.
4. Click any point in scatter plot, or move Sample slider, to inspect one sample in detail.